# 22. YOLOv8n-seg 잎/줄기 카운터 학습

**입력**: 21번 노트북에서 자동 생성한 `yolo_world_labels/` (leaf/stem 폴리곤 + 빈 파일)

**출력**: 잎·줄기 인스턴스를 세는 YOLOv8n-seg 모델

- 클래스 0 = `leaf`, 클래스 1 = `stem`
- 빈 txt 파일(흙만 있는 사진)도 포함해 배경 판별력을 높임

<a href="https://colab.research.google.com/github/gimme-water/gimme-water-ML/blob/main/notebooks/22_train_yolov8_seg.ipynb" target="_parent">
<img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import os, shutil, warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import cv2
from pathlib import Path
from PIL import Image

warnings.filterwarnings('ignore')

try:
    from google.colab import drive
    IN_COLAB = True
    ROOT = Path('/content')
except ImportError:
    IN_COLAB = False
    ROOT = Path(os.getcwd()).parent

if IN_COLAB:
    drive.mount('/content/drive')
    PACK_DIR = Path('/content/drive/My Drive/colab_drive/dataset.zip')
    DATA_DIR = Path('/content/dataset')
    shutil.copy(PACK_DIR, '/content/')
    !unzip -o -q /content/dataset.zip -d {DATA_DIR}

DATASET_DIR     = ROOT / 'dataset' / 'collector'
AUTO_LABELS_DIR = DATASET_DIR / 'yolo_world_labels'

print(f'ROOT           : {ROOT}')
print(f'AUTO_LABELS_DIR: {AUTO_LABELS_DIR}')

ROOT           : /Users/mainframe/Workspace/Graduates/planta-gochi-ML
AUTO_LABELS_DIR: /Users/mainframe/Workspace/Graduates/planta-gochi-ML/dataset/collector/yolo_world_labels


## 1. 데이터셋 준비

In [2]:
# parquet 로드 & 이미지 stem → row 매핑
splits = ['train', 'val', 'test']
df_all = pd.concat(
    [pd.read_parquet(DATASET_DIR / f'{s}.parquet') for s in splits],
    ignore_index=True,
)

df_all.head(5)

,air_temperature,air_humidity,light,soil_temperature,soil_moisture,soil_ph,soil_ec,soil_nitrogen,soil_phosphorus,soil_potassium,image_filename,delta_days,split
0,25.3,25.5,3297.0,23.1,17.7,7.2,175.0,7.0,12.0,28.0,collector_001_20260420215114.jpg,0.000000,train
1,25.3,25.4,3235.0,23.1,17.7,7.2,175.0,7.0,12.0,28.0,collector_001_20260420215157.jpg,0.000498,train
2,25.8,25.3,2893.0,23.1,17.8,7.2,176.0,7.0,12.0,28.0,collector_001_20260420215449.jpg,0.002488,train
3,21.0,26.1,93.0,23.1,17.6,7.3,175.0,7.0,12.0,28.0,collector_001_20260420220126.jpg,0.007083,train
4,21.5,21.8,1.0,22.7,17.9,7.3,180.0,8.0,12.0,29.0,collector_001_20260420220628.jpg,0.010579,train


In [3]:
stem_to_row = {Path(r['image_filename']).stem: r for _, r in df_all.iterrows()}

# 라벨 파일 목록
all_txts  = sorted(AUTO_LABELS_DIR.glob('*.txt'))
nonempty  = [t for t in all_txts if t.stat().st_size > 0]
empty     = [t for t in all_txts if t.stat().st_size == 0]

print(f'라벨 있음(leaf/stem): {len(nonempty)}')
print(f'빈 라벨(흙/배경)   : {len(empty)}')
print(f'합계               : {len(all_txts)}')

라벨 있음(leaf/stem): 2517
빈 라벨(흙/배경)   : 916
합계               : 3433


In [8]:
# ── YOLO 디렉토리 구조 생성 ───────────────────────────────────────────────────
# DATASET_DIR/{split}/photos/*.jpg
# DATASET_DIR/{split}/labels/*.txt

for s in splits:
    (DATASET_DIR / s / 'photos').mkdir(parents=True, exist_ok=True)
    (DATASET_DIR / s / 'labels').mkdir(parents=True, exist_ok=True)

skipped = 0
copied  = {s: 0 for s in splits}

for txt_path in all_txts:
    stem = txt_path.stem
    row  = stem_to_row.get(stem)
    if row is None:
        print(row)
        skipped += 1
        continue

    split    = row['split']
    lbl_dst  = DATASET_DIR / split / 'labels' / txt_path.name

    # if not img_dst.exists():
    #     shutil.copy2(img_src, img_dst)
    if not lbl_dst.exists():
        shutil.copy2(txt_path, lbl_dst)

    copied[split] += 1

print('복사 완료:')
for s, n in copied.items():
    print(f'  {s:5s}: {n}장')
if skipped:
    print(f'  parquet 미매칭 스킵: {skipped}')

복사 완료:
  train: 2350장
  val  : 666장
  test : 417장


In [9]:
import yaml

yaml_cfg = {
    'path' : str(DATASET_DIR),
    'train': 'train/photos',
    'val'  : 'val/photos',
    'test' : 'test/photos',
    'nc'   : 2,
    'names': ['leaf', 'stem'],
}
yaml_path = DATASET_DIR / 'dataset.yaml'
with open(yaml_path, 'w') as f:
    yaml.dump(yaml_cfg, f, default_flow_style=False, allow_unicode=True)

print(f'dataset.yaml 저장: {yaml_path}')
print(yaml.dump(yaml_cfg, default_flow_style=False))

dataset.yaml 저장: /Users/mainframe/Workspace/Graduates/planta-gochi-ML/dataset/collector/dataset.yaml
names:
- leaf
- stem
nc: 2
path: /Users/mainframe/Workspace/Graduates/planta-gochi-ML/dataset/collector
test: test/photos
train: train/photos
val: val/photos



In [10]:
# 분포 확인
for s in splits:
    imgs  = list((DATASET_DIR / s / 'photos').glob('*'))
    lbls  = list((DATASET_DIR / s / 'labels').glob('*.txt'))
    empties = sum(1 for l in lbls if l.stat().st_size == 0)
    print(f'{s:5s}  이미지={len(imgs):4d}  라벨={len(lbls):4d}  (빈={empties} 흙/배경)')

train  이미지=3742  라벨=2350  (빈=892 흙/배경)
val    이미지= 802  라벨= 666  (빈=7 흙/배경)
test   이미지= 803  라벨= 417  (빈=17 흙/배경)


## 2. 학습

In [11]:

# ── 1) 손상 이미지 제거  2) 라벨 좌표 정제 ───────────────────────────────────
import warnings as _w
import os, tempfile

removed_imgs   = {s: 0 for s in splits}
fixed_labels   = 0
removed_polys  = 0

def is_valid_image(img_path: Path) -> bool:
    # 검사 1: JPEG EOF 마커 (0xFFD9)
    try:
        with open(img_path, 'rb') as f:
            if f.read()[-2:] != b'\xff\xd9':
                return False
    except Exception:
        return False

    # 검사 2: PIL 픽셀 디코딩
    try:
        with _w.catch_warnings():
            _w.simplefilter('ignore')
            im = Image.open(img_path)
            im.load()
        if im.size[0] == 0 or im.size[1] == 0:
            return False
    except Exception:
        return False

    # 검사 3: OpenCV — C 레벨 stderr 캡처로 "Corrupt JPEG" 경고 감지
    # cv2.imread 는 손상 JPEG도 None 대신 partial array 를 반환하므로
    # 반환값이 아니라 libjpeg 가 출력하는 경고 문자열을 직접 잡아야 함
    tmp_fd, tmp_name = tempfile.mkstemp()
    saved_fd = os.dup(2)
    try:
        os.dup2(tmp_fd, 2)
        img_cv = cv2.imread(str(img_path), cv2.IMREAD_COLOR)
    finally:
        os.dup2(saved_fd, 2)
        os.close(saved_fd)
        os.close(tmp_fd)

    try:
        with open(tmp_name, 'r', errors='ignore') as f:
            cv_stderr = f.read()
        os.unlink(tmp_name)
    except Exception:
        cv_stderr = ''

    if 'Corrupt JPEG' in cv_stderr or img_cv is None or img_cv.size == 0:
        return False

    return True

for s in splits:
    img_dir = DATASET_DIR / s / 'photos'
    lbl_dir = DATASET_DIR / s / 'labels'

    for img_path in sorted(img_dir.glob('*')):
        if not is_valid_image(img_path):
            lbl_path = lbl_dir / (img_path.stem + '.txt')
            img_path.unlink(missing_ok=True)
            lbl_path.unlink(missing_ok=True)
            removed_imgs[s] += 1
            continue

        # ── 라벨: 좌표 클리핑 + 퇴화 폴리곤 제거 ───────────────────────────
        lbl_path = lbl_dir / (img_path.stem + '.txt')
        if not lbl_path.exists() or lbl_path.stat().st_size == 0:
            continue

        lines_in  = lbl_path.read_text().strip().splitlines()
        lines_out = []
        changed   = False

        for line in lines_in:
            parts = line.strip().split()
            if len(parts) < 7:
                changed = True
                removed_polys += 1
                continue

            cls_id  = parts[0]
            coords  = np.array(parts[1:], dtype=float)
            clipped = np.clip(coords, 0.0, 1.0)
            if not np.array_equal(coords, clipped):
                changed = True

            pts = clipped.reshape(-1, 2)
            if len(pts) < 3:
                changed = True
                removed_polys += 1
                continue

            if (pts[:, 0].max() - pts[:, 0].min() == 0 or
                    pts[:, 1].max() - pts[:, 1].min() == 0):
                changed = True
                removed_polys += 1
                continue

            lines_out.append(f'{cls_id} ' + ' '.join(f'{v:.6f}' for v in clipped))

        if changed:
            lbl_path.write_text('\n'.join(lines_out) + ('\n' if lines_out else ''))
            fixed_labels += 1

print('=== 검증 완료 ===')
print('손상 이미지 제거:')
for s, n in removed_imgs.items():
    remaining = len(list((DATASET_DIR / s / 'photos').glob('*')))
    print(f'  {s:5s}: 제거 {n}장  남음 {remaining}장')
print(f'라벨 정제: {fixed_labels}개 파일 수정, {removed_polys}개 폴리곤 제거')


=== 검증 완료 ===
손상 이미지 제거:
  train: 제거 9장  남음 3733장
  val  : 제거 0장  남음 802장
  test : 제거 0장  남음 803장
라벨 정제: 0개 파일 수정, 0개 폴리곤 제거


In [ ]:
from ultralytics import YOLO
import torch

DEVICE = 'cuda' if torch.cuda.is_available() else 'mps' if torch.backends.mps.is_available() else 'cpu'
print(f'Device: {DEVICE}')

model = YOLO('yolov8n-seg.pt')

results = model.train(
    data    = str(yaml_path),
    epochs  = 100,
    imgsz   = 640,
    batch   = 16,
    device  = DEVICE,
    project = str(ROOT / 'runs' / 'seg'),
    name    = 'leaf_stem_v1',
    patience= 20,
    exist_ok= True,
    verbose = True,
)

Device: mps
New https://pypi.org/project/ultralytics/8.4.52 available 😃 Update with 'pip install -U ultralytics'
Ultralytics 8.4.51 🚀 Python-3.13.11 torch-2.11.0 MPS (Apple M4 Max)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/Users/mainframe/Workspace/Graduates/planta-gochi-ML/dataset/collector/dataset.yaml, degrees=0.0, deterministic=True, device=mps, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=100, erasing=0.4, exist_ok=True, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolov8n-seg.p

## 3. 평가

In [ ]:
best_weights = ROOT / 'runs' / 'seg' / 'leaf_stem_v1' / 'weights' / 'best.pt'
model_best   = YOLO(str(best_weights))

metrics = model_best.val(data=str(yaml_path), split='test', verbose=True)

print(f'\nmAP50     : {metrics.seg.map50:.4f}')
print(f'mAP50-95  : {metrics.seg.map:.4f}')
print(f'Precision : {metrics.seg.mp:.4f}')
print(f'Recall    : {metrics.seg.mr:.4f}')

In [ ]:
# 학습 곡선 시각화
results_csv = ROOT / 'runs' / 'seg' / 'leaf_stem_v1' / 'results.csv'
df_res = pd.read_csv(results_csv)
df_res.columns = df_res.columns.str.strip()

fig, axes = plt.subplots(1, 3, figsize=(15, 4))

axes[0].plot(df_res['epoch'], df_res['train/seg_loss'], label='train')
axes[0].plot(df_res['epoch'], df_res['val/seg_loss'],   label='val')
axes[0].set_title('Seg Loss')
axes[0].legend()

axes[1].plot(df_res['epoch'], df_res['metrics/mAP50(M)'],    label='mAP50')
axes[1].plot(df_res['epoch'], df_res['metrics/mAP50-95(M)'], label='mAP50-95')
axes[1].set_title('mAP (mask)')
axes[1].legend()

axes[2].plot(df_res['epoch'], df_res['metrics/precision(M)'], label='precision')
axes[2].plot(df_res['epoch'], df_res['metrics/recall(M)'],    label='recall')
axes[2].set_title('Precision / Recall')
axes[2].legend()

for ax in axes:
    ax.set_xlabel('epoch')
    ax.grid(alpha=0.3)

fig.tight_layout()
plt.show()

## 4. 추론 & 카운트 확인

In [ ]:
CLASS_NAMES = ['leaf', 'stem']
COLORS      = plt.cm.tab10.colors

def count_instances(result) -> dict:
    """클래스별 인스턴스 수 반환."""
    if result.boxes is None or len(result.boxes) == 0:
        return {n: 0 for n in CLASS_NAMES}
    cls_ids = result.boxes.cls.cpu().numpy().astype(int)
    counts  = {n: 0 for n in CLASS_NAMES}
    for cid in cls_ids:
        if cid < len(CLASS_NAMES):
            counts[CLASS_NAMES[cid]] += 1
    return counts


def visualize_prediction(image_rgb, result, figsize=(11, 6)):
    """마스크 오버레이 + 카운트 표시."""
    display_img = image_rgb.copy().astype(np.float32)
    h, w = image_rgb.shape[:2]

    has_masks = result.masks is not None and len(result.masks) > 0
    if has_masks:
        cls_ids = result.boxes.cls.cpu().numpy().astype(int)
        confs   = result.boxes.conf.cpu().numpy()
        masks   = result.masks.data.cpu().numpy()  # (N, H', W')

        for i, (mask_small, cid, conf) in enumerate(zip(masks, cls_ids, confs)):
            # 마스크를 원본 해상도로 리사이즈
            mask = cv2.resize(mask_small, (w, h), interpolation=cv2.INTER_LINEAR) > 0.5
            color = COLORS[cid % len(COLORS)]
            c255  = np.array([color[0]*255, color[1]*255, color[2]*255], dtype=np.float32)
            display_img[mask] = 0.55 * display_img[mask] + 0.45 * c255

    display_img = np.clip(display_img, 0, 255).astype(np.uint8)

    fig, ax = plt.subplots(figsize=figsize)
    ax.imshow(display_img)
    ax.axis('off')

    if has_masks:
        for i, (mask_small, cid, conf) in enumerate(zip(masks, cls_ids, confs)):
            mask   = cv2.resize(mask_small, (w, h), interpolation=cv2.INTER_LINEAR) > 0.5
            color  = COLORS[cid % len(COLORS)]
            m_uint = mask.astype(np.uint8)
            contours, _ = cv2.findContours(m_uint, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
            for c in contours:
                pts = c.reshape(-1, 2)
                xs  = np.append(pts[:, 0], pts[0, 0])
                ys  = np.append(pts[:, 1], pts[0, 1])
                ax.plot(xs, ys, color='white', lw=4,   zorder=3, solid_capstyle='round')
                ax.plot(xs, ys, color=color,   lw=2,   zorder=4, solid_capstyle='round')

            ys_m, xs_m = np.where(mask)
            if len(xs_m):
                ax.text(
                    int(xs_m.mean()), int(ys_m.mean()),
                    f"{CLASS_NAMES[cid] if cid < len(CLASS_NAMES) else cid}\n{conf:.2f}",
                    color='white', fontsize=8, fontweight='bold',
                    ha='center', va='center', zorder=7,
                    bbox=dict(boxstyle='round,pad=0.2',
                              fc=COLORS[cid % len(COLORS)], alpha=0.75, lw=0),
                )

    counts  = count_instances(result)
    cnt_str = '  '.join(f'{k}: {v}' for k, v in counts.items())
    ax.set_title(cnt_str, fontsize=11, pad=6,
                 bbox=dict(boxstyle='round,pad=0.3', fc='white', alpha=0.8))

    patches = [
        mpatches.Patch(color=COLORS[i % len(COLORS)], label=name)
        for i, name in enumerate(CLASS_NAMES)
    ]
    ax.legend(handles=patches, loc='upper right', fontsize=9, framealpha=0.85)
    fig.tight_layout(pad=0.4)
    return fig


print('추론 헬퍼 정의 완료')

In [ ]:
# 테스트셋 무작위 샘플 추론
import random

N_INFER = 12
test_images = sorted((DATASET_DIR / 'test' / 'photos').glob('*'))
samples     = random.sample(test_images, min(N_INFER, len(test_images)))

for img_path in samples:
    image_rgb = np.array(Image.open(img_path).convert('RGB'))
    result    = model_best.predict(image_rgb, verbose=False)[0]
    counts    = count_instances(result)

    fig = visualize_prediction(image_rgb, result)
    fig.suptitle(img_path.name, fontsize=9, y=1.01)
    plt.show()
    plt.close(fig)

    print(f'  {img_path.name}: leaf={counts["leaf"]}  stem={counts["stem"]}')

In [ ]:
# 흙(빈) 이미지 추론 — false positive 확인용
empty_labels = [p for p in (DATASET_DIR / 'test' / 'labels').glob('*.txt')
                if p.stat().st_size == 0]
print(f'테스트셋 빈 이미지 수: {len(empty_labels)}')

fp_total = 0
for lbl_path in random.sample(empty_labels, min(6, len(empty_labels))):
    img_path  = DATASET_DIR / 'test' / 'photos' / (lbl_path.stem + '.jpg')
    if not img_path.exists():
        continue
    image_rgb = np.array(Image.open(img_path).convert('RGB'))
    result    = model_best.predict(image_rgb, verbose=False)[0]
    counts    = count_instances(result)
    fp        = sum(counts.values())
    fp_total += fp

    fig = visualize_prediction(image_rgb, result)
    fig.suptitle(f'{img_path.name}  [정답: 흙/없음]  FP={fp}', fontsize=9, y=1.01)
    plt.show()
    plt.close(fig)

print(f'\n흙 이미지 총 false positive: {fp_total}')